In [ ]:
!pip uninstall -y flash-attn
!pip install "transformers<4.47.0" "peft<0.14.0" "ms-swift[llm]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 298.8/298.8 kB 29.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of trl to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of trl to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 kB 11.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!unzip images.zip

Archive:  images.zip
   creating: images/
  inflating: images/image478.jpg     
  inflating: __MACOSX/images/._image478.jpg  
  inflating: images/image336.jpg     
  inflating: __MACOSX/images/._image336.jpg  
  inflating: images/image450.jpg     
  inflating: __MACOSX/images/._image450.jpg  
  inflating: images/image444.jpg     
  inflating: __MACOSX/images/._image444.jpg  
  inflating: images/image322.jpg     
  inflating: __MACOSX/images/._image322.jpg  
  inflating: images/image493.jpg     
  inflating: __MACOSX/images/._image493.jpg  
  inflating: images/image487.jpg     
  inflating: __MACOSX/images/._image487.jpg  
  inflating: images/image108.jpg     
  inflating: __MACOSX/images/._image108.jpg  
  inflating: images/image652.jpg     
  inflating: __MACOSX/images/._image652.jpg  
  inflating: images/image134.jpg     
  inflating: __MACOSX/images/._image134.jpg  
  inflating: images/image120.jpg     
  inflating: __MACOSX/images/._image120.jpg  
  inflating: images/image646.jpg  

In [ ]:
import json
def build_jsonl(input_file,output_file):
  data = json.load(open(input_file))
  with open(output_file,'w',encoding = 'utf-8') as f:
    for item in data:
      for entry in item["conversations"]:
        if entry["from"] == "assistant" and isinstance(entry["value"], dict):
          entry["value"] = json.dumps(entry["value"], ensure_ascii=False)
      json_record = json.dumps(item,ensure_ascii=False)
      f.write(json_record + '\n')
  print(f"✅ Done! {len(data)} samples converted to {output_file}")

In [ ]:
build_jsonl('Fine_Tune1.json','FineTune.jsonl')

✅ Done! 1003 samples converted to FineTune.jsonl


In [ ]:
import json
from sklearn.model_selection import train_test_split
from collections import Counter

with open('FineTune.jsonl', 'r') as f:
    lines = f.readlines()
data = [json.loads(line) for line in lines]

stratify_labels = []
for item in data:
    try:
        content = item['conversations'][1]['value']
        if isinstance(content, str):
            val = json.loads(content)
        else:
            val = content
        t = val['labels'].get('type', 'none')
        stratify_labels.append(t)
    except:
        stratify_labels.append('none')

counts = Counter(stratify_labels)
main_indices = [i for i, l in enumerate(stratify_labels) if counts[l] > 1]
loner_indices = [i for i, l in enumerate(stratify_labels) if counts[l] <= 1]

train_idx, test_idx = train_test_split(
    main_indices, test_size=0.20, stratify=[stratify_labels[i] for i in main_indices], random_state=42
)

train_data = [data[i] for i in train_idx] + [data[i] for i in loner_indices]
test_data = [data[i] for i in test_idx]

with open('train_split.jsonl', 'w') as f:
    for item in train_data: f.write(json.dumps(item) + '\n')
with open('test_split.jsonl', 'w') as f:
    for item in test_data: f.write(json.dumps(item) + '\n')

print(f"✅ Stratified Split Done! Train: {len(train_data)} | Test: {len(test_data)}")

✅ Stratified Split Done! Train: 802 | Test: 201


In [ ]:
!pip install huggingface_hub
from huggingface_hub import snapshot_download
snapshot_download(repo_id="OpenGVLab/InternVL2_5-8B", local_dir="internvl2_5_8b", local_dir_use_symlinks=False)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:986: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 28 files:   0%|          | 0/28 [00:00<?, ?it/s]

conversation.py: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

configuration_internvl_chat.py: 0.00B [00:00, ?B/s]

...

tokenization_internlm2.py: 0.00B [00:00, ?B/s]

tokenization_internlm2_fast.py: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/1.48M [00:00<?, ?B/s]

'/content/internvl2_5_8b'

In [ ]:
from huggingface_hub import snapshot_download
import os

sft_adapter_path = snapshot_download(
    repo_id="debarghaNath/BullyingMemeDetector",
    allow_patterns=["1000_32_best_model/*"],
    local_dir="/content/sft_adapter"
)

# 2. Download the DPO Adapter (The one that knows the Expert Logic)
dpo_adapter_path = snapshot_download(
    repo_id="debarghaNath/BullyingMemeDetector",
    allow_patterns=["DPO_v2_best_model/*"],
    local_dir="/content/dpo_adapter"
)

print(f"✅ SFT Adapter ready at: {sft_adapter_path}")
print(f"✅ DPO Adapter ready at: {dpo_adapter_path}")


!mv /content/sft_adapter/1000_32_best_model/* /content/sft_adapter/ 2>/dev/null
!mv /content/dpo_adapter/DPO_v2_best_model/* /content/dpo_adapter/ 2>/dev/null

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

1000_32_best_model/adapter_model.safeten(…):   0%|          | 0.00/314M [00:00<?, ?B/s]

1000_32_best_model/scheduler.pt:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

1000_32_best_model/rng_state.pth:   0%|          | 0.00/14.6k [00:00<?, ?B/s]

1000_32_best_model/optimizer.pt:   0%|          | 0.00/629M [00:00<?, ?B/s]

...

args.json: 0.00B [00:00, ?B/s]

adapter_config.json:   0%|          | 0.00/672 [00:00<?, ?B/s]

additional_config.json:   0%|          | 0.00/67.0 [00:00<?, ?B/s]

DPO_v2_best_model/training_args.bin:   0%|          | 0.00/7.89k [00:00<?, ?B/s]

✅ SFT Adapter ready at: /content/sft_adapter
✅ DPO Adapter ready at: /content/dpo_adapter


In [ ]:

dpo_adapter_path = snapshot_download(
    repo_id="debarghaNath/BullyingMemeDetector",
    allow_patterns=["DPO_v2_last_model/*"],
    local_dir="/content/dpo_adapter_last"
)

print(f"✅ SFT Adapter ready at: {sft_adapter_path}")
print(f"✅ DPO Adapter ready at: {dpo_adapter_path}")


!mv /content/dpo_adapter_last/DPO_v2_lastt_model/* /content/dpo_adapter_last/ 2>/dev/null

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

DPO_v2_last_model/scheduler.pt:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

DPO_v2_last_model/optimizer.pt:   0%|          | 0.00/157M [00:00<?, ?B/s]

DPO_v2_last_model/adapter_model.safetens(…):   0%|          | 0.00/78.6M [00:00<?, ?B/s]

DPO_v2_last_model/rng_state.pth:   0%|          | 0.00/14.6k [00:00<?, ?B/s]

...

additional_config.json:   0%|          | 0.00/67.0 [00:00<?, ?B/s]

trainer_state.json: 0.00B [00:00, ?B/s]

args.json: 0.00B [00:00, ?B/s]

DPO_v2_last_model/training_args.bin:   0%|          | 0.00/7.89k [00:00<?, ?B/s]

✅ SFT Adapter ready at: /content/sft_adapter
✅ DPO Adapter ready at: /content/dpo_adapter_last


In [ ]:
!mv /content/dpo_adapter_last/DPO_v2_last_model/* /content/dpo_adapter_last/ 2>/dev/null

In [ ]:
!swift export \
    --model /content/internvl2_5_8b \
    --model_type internvl_chat \
    --adapters /content/sft_adapter \
    --merge_lora true \
    --output_dir /content/model_sft_merged

2026-03-31 08:13:07.209824: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-31 08:13:07.227580: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774944787.249515    5223 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774944787.256784    5223 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774944787.275259    5223 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
!swift export \
    --model /content/model_sft_merged \
    --model_type internvl_chat \
    --adapters /content/dpo_adapter_last \
    --merge_lora true \
    --output_dir /content/MemeExpert_DPO_FINAL_ULTIMATE_last

2026-03-31 09:01:31.793184: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774947691.815495   19595 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774947691.822953   19595 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774947691.841729   19595 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774947691.841755   19595 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774947691.841758   19595 computation_placer.cc:177] computation placer alr

In [ ]:
import torch
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
import os
import torch
from swift.infer_engine import TransformersEngine, InferRequest, RequestConfig

print("🚀 Initializing Meme Expert (v4 Modern Architecture)...")

model_dir = "/content/MemeExpert_DPO_FINAL_ULTIMATE_last"

engine = TransformersEngine(
    model_dir,
    model_type='internvl_chat',
    model_kwargs={
        'attn_implementation': 'eager',
        'torch_dtype': torch.bfloat16,
        'device_map': 'auto'
    }
)

request_config = RequestConfig(
    max_tokens=512,
    temperature=0,
    top_p=0.7
)
def analyze_meme(image_path):
    """Performs multimodal bullying analysis using the v4 Engine."""
    if not os.path.exists(image_path):
        return f"❌ Error: {image_path} not found."

    system_rules = (
        "You are an expert in sociolinguistics and multimodal harm analysis. "
        "You MUST select labels ONLY from the following closed sets: "
        "bullying = [Yes, No], "
        "target = [individual, individual(multiple) ,demographic_group, organization, occupational_group, none]; "
        "mechanism = [sarcastic_dissonance, violent_juxtaposition, normative_comparison, coded_symbolism, metaphorical_dehumanization, punching_down, other, none]; "
        "type = [relational_social, gender_based, identity_based, physical_appearance, cognitive_intellectual, religious_political, other, none]; "
        "severity = [none, low, medium, high]. Rules: Do NOT invent new labels. "
        "Do NOT explain labels outside the JSON. If uncertain, choose the closest label from the list. "
        "Output valid JSON only."
    )
    query = f"<img>{image_path}</img>\nPerform a multimodal bullying analysis on this content."

    infer_request = InferRequest(
        messages=[
            {'role': 'system', 'content': system_rules},
            {'role': 'user', 'content': query}
        ],
        images=[image_path]
    )

    resp_list = engine.infer([infer_request], request_config)
    return resp_list[0].choices[0].message.content



[INFO:swift] Setting torch_dtype: torch.bfloat16
[INFO:swift] model_kwargs: {'attn_implementation': 'eager', 'torch_dtype': torch.bfloat16, 'device_map': 'cuda:0'}


🚀 Initializing Meme Expert (v4 Modern Architecture)...
FlashAttention2 is not installed.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

[INFO:swift] Create the template for the infer_engine
[INFO:swift] Successfully loaded /content/MemeExpert_DPO_FINAL_ULTIMATE_last/args.json.
[INFO:swift] default_system: '你是书生·万象，英文名是InternVL，是由上海人工智能实验室、清华大学及多家合作单位联合开发的多模态大语言模型。'
[INFO:swift] max_length: 32768
[INFO:swift] response_prefix: ''
[INFO:swift] agent_template: react_en
[INFO:swift] norm_bbox: norm1000


In [ ]:
with open('test_split.jsonl', 'r') as f:
    test_items = [json.loads(line) for line in f]

In [ ]:
import json
import pandas as pd
import re 
from tqdm import tqdm

results_list = []

print(f"🚀 Starting final evaluation on {len(test_items)} images...")

for item in tqdm(test_items):
    test_img = item['images'][0]

    try:
        raw_output = analyze_meme(test_img)
        match = re.search(r'\{.*\}', raw_output, re.DOTALL)

        if not match:
            raise ValueError("No JSON found in model output")

        json_str = match.group(0)

        data = json.loads(json_str)

        if 'labels' in data:
            labels = data.get('labels', {})
            analysis = data.get('analysis', {})
        else:
            labels = data
            analysis = data

        results_list.append({
            "image_path": test_img,
            "pred_bullying": labels.get('bullying'),
            "pred_target": labels.get('target'),
            "pred_mechanism": labels.get('mechanism'),
            "pred_type": labels.get('type'),
            "pred_severity": labels.get('severity'),
            "visual_cues": analysis.get('visual_cues', 'N/A'),
            "text_cues": analysis.get('text_cues', 'N/A'),
            "interpretation": analysis.get('interpretation', 'N/A')
        })

    except Exception as e:
        results_list.append({
            "image_path": test_img,
            "pred_bullying": "ERROR",
            "interpretation": f"Failed to parse: {str(e)}",
            "raw_text_debug": raw_output[:100]
        })

df = pd.DataFrame(results_list)
df.to_csv("bullying_analysis_results.csv", index=False)
print("\n✅ Loop Complete! Data saved to bullying_analysis_results.csv")

🚀 Starting final evaluation on 201 images...


100%|██████████| 201/201 [38:11<00:00, 11.40s/it]


✅ Loop Complete! Data saved to bullying_analysis_results.csv


In [ ]:
result = analyze_meme("/content/image1004.jpg")
print(result)

{"analysis": {"visual_cues": "A photograph of an older man in a convenience store, viewed from behind. He is wearing a blue puffer jacket and light blue jeans. The man is standing in a way that his backside is prominently visible, showing a large, exaggerated buttocks shape.", "text_cues": "The overlay text reads: 'Bro have unnatural legs to torso ratio lol' accompanied by monkey covering eyes and laughing emojis.", "text_location": "embedded_in_image", "interpretation": "This content targets an 'individual' through 'physical_appearance' shaming. By using the term 'unnatural' and a 'ratios' comparison, the meme implies that the man's body shape is a result of cosmetic surgery or a medical condition that is socially undesirable. The emojis (monkey covering eyes and laughing) serve to mock the subject's anatomy, framing his physical traits as a source of ridicule. This is a form of 'physical_appearance' bullying that uses normative comparison to humiliate the subject.", "confidence": "hi